# NB22 — Poisoned Edge, Dependency Trace, and Complete Reversal

**MFin laboratory companion**  
This notebook deliberately approves one wrong relationship, allows it to influence several later decisions, traces the complete dependency cone, and reverses every contaminated object and edge. The lesson is precise: rollback is not deletion of the root error. It is dependency repair plus proof that no active canonical state still depends on the error.

## Learning objectives

By the end, students should be able to:

1. distinguish an approved edge from a true edge;
2. define which relationship types transmit dependency;
3. enumerate every downstream decision and derived object affected by a root error;
4. demote contaminated state to archival rather than silently delete it;
5. produce a reversal ledger and a restored-state proof; and
6. explain why a graph without dependency semantics cannot make reversibility operational.

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def add_node(g, node_id, **attrs):
    g['nodes'][node_id] = attrs

def add_edge(g, edge_id, source, target, **attrs):
    g['edges'][edge_id] = {'source': source, 'target': target, **attrs}

def active_projection(g):
    nodes = {n: a for n, a in g['nodes'].items() if a.get('governance_status') == 'canonical'}
    edges = {k: a for k, a in g['edges'].items()
             if a['source'] in nodes and a['target'] in nodes
             and a.get('governance_status') == 'canonical'}
    return {'nodes': nodes, 'edges': edges}

def graph_hash(g):
    nodes = sorted((n, sorted(a.items())) for n, a in g['nodes'].items())
    edges = sorted((k, sorted(a.items())) for k, a in g['edges'].items())
    payload = json.dumps({'nodes': nodes, 'edges': edges}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()

DEPENDENCY_TYPES = {'supports', 'assumption_for', 'informs', 'derived_from'}

## 1. Freeze the clean baseline

The baseline contains two admitted source artifacts and one company. It contains no clearance claim and no decision that depends on such a claim.

In [ ]:
G = {'graph_version': 'NB22-clean-v1', 'nodes': {}, 'edges': {}}

add_node(G, 'SRC_LAWYER_MEMO', object_type='source_artifact', title='Counsel memo — clearance pending',
           epistemic_class='asserted-by-source', governance_status='canonical', access_profile='deal_team')
add_node(G, 'SRC_STATUS_EMAIL', object_type='source_artifact', title='Status email — no regulator decision',
           epistemic_class='observed', governance_status='canonical', access_profile='deal_team')
add_node(G, 'ENTITY_TARGET', object_type='entity', title='Target Payments Ltd.',
           epistemic_class='observed', governance_status='canonical', access_profile='deal_team')
add_node(G, 'OBS_MARKET_SHARE', object_type='observation', title='Target market share: 8%',
         epistemic_class='observed', governance_status='canonical', access_profile='deal_team')
add_edge(G, 'OBS_1', 'ENTITY_TARGET', 'OBS_MARKET_SHARE', relationship_type='describes', governance_status='canonical')

BASELINE_HASH = graph_hash(active_projection(G))
BASELINE_HASH

## 2. Deliberately approve the wrong edge

The laboratory now commits an incident. A model derives the false claim **“regulatory clearance secured.”** A reviewer approves a `supports` edge from a memo that actually says clearance is pending. The approval is procedurally valid and epistemically wrong.

**Before executing the next cell, write your prediction:** Which later objects will become contaminated, and which seemingly related objects should remain outside the dependency cone?

In [ ]:
add_node(G, 'CLAIM_CLEARANCE', object_type='claim', title='Regulatory clearance secured',
           epistemic_class='derived-by-model', governance_status='canonical', access_profile='deal_team')

add_edge(G, 'EDGE_POISON', 'SRC_LAWYER_MEMO', 'CLAIM_CLEARANCE',
           relationship_type='supports', governance_status='canonical',
           approval_record='APPROVAL_017', approved_by='Reviewer_A', approved_at=utc_now(),
           poisoned=True)

# Decisions and derived objects admitted after the poisoned edge.
downstream_nodes = {
    'DECISION_ACQUIRE': ('decision', 'Approve acquisition'),
    'DECISION_FINANCE': ('decision', 'Draw acquisition facility'),
    'PLAN_INTEGRATION': ('plan', 'Launch integration programme'),
    'DECISION_HEDGE': ('decision', 'Enter FX hedge for purchase price'),
}
for node_id, (kind, title) in downstream_nodes.items():
    add_node(G, node_id, object_type=kind, title=title,
               epistemic_class='inferred-under-assumption', governance_status='canonical',
               access_profile='deal_team')

add_edge(G, 'DEP_1', 'CLAIM_CLEARANCE', 'DECISION_ACQUIRE', relationship_type='assumption_for', governance_status='canonical')
add_edge(G, 'DEP_2', 'DECISION_ACQUIRE', 'DECISION_FINANCE', relationship_type='informs', governance_status='canonical')
add_edge(G, 'DEP_3', 'DECISION_ACQUIRE', 'PLAN_INTEGRATION', relationship_type='informs', governance_status='canonical')
add_edge(G, 'DEP_4', 'DECISION_FINANCE', 'DECISION_HEDGE', relationship_type='informs', governance_status='canonical')

# The nearby baseline observation remains independent and must not be reversed.
print('Active nodes after contamination:', len(active_projection(G)['nodes']))
print('Active edges after contamination:', len(active_projection(G)['edges']))

## 3. Trace the dependency cone

Only declared dependency-transmitting relationship types are traversed. Physical proximity, semantic similarity, or a shared entity is not enough to justify reversal.

In [ ]:
def dependency_cone(g, root_node, dependency_types=DEPENDENCY_TYPES):
    contaminated = {root_node}
    frontier = [root_node]
    traversed_edges = []
    while frontier:
        current = frontier.pop(0)
        for key, attrs in g['edges'].items():
            if attrs['source'] != current:
                continue
            target = attrs['target']
            if attrs.get('governance_status') != 'canonical':
                continue
            if attrs.get('relationship_type') not in dependency_types:
                continue
            traversed_edges.append((current, target, key, attrs.get('relationship_type')))
            if target not in contaminated:
                contaminated.add(target)
                frontier.append(target)
    return contaminated, traversed_edges

affected_nodes, affected_edges = dependency_cone(G, 'CLAIM_CLEARANCE')
pprint(affected_edges)

In [ ]:
affected_decisions = sorted(
    n for n in affected_nodes if G['nodes'][n].get('object_type') == 'decision'
)
print('Affected nodes:', sorted(affected_nodes))
print('Affected decisions:', affected_decisions)
assert affected_decisions == ['DECISION_ACQUIRE', 'DECISION_FINANCE', 'DECISION_HEDGE']
assert 'OBS_MARKET_SHARE' not in affected_nodes

## 4. Reverse the lot

Reversal archives the poisoned edge, the false claim, every dependency-transmitting downstream edge, and every dependent object. Nothing is silently deleted. The ledger preserves what happened and who must reconcile any external action.

In [ ]:
def reverse_dependency_cone(g, edge_key='EDGE_POISON'):
    source = g['edges'][edge_key]['source']
    root = g['edges'][edge_key]['target']
    affected_nodes, affected_edges = dependency_cone(g, root)
    reversed_at = utc_now()

    g['edges'][edge_key]['governance_status'] = 'archived'
    g['edges'][edge_key]['reversal_reason'] = 'approved-but-wrong relationship'
    g['edges'][edge_key]['reversed_at'] = reversed_at

    for node_id in affected_nodes:
        g['nodes'][node_id]['governance_status'] = 'archived'
        g['nodes'][node_id]['reversal_reason'] = f'depends on {edge_key}'
        g['nodes'][node_id]['reversed_at'] = reversed_at

    for u, v, key, _ in affected_edges:
        g['edges'][key]['governance_status'] = 'archived'
        g['edges'][key]['reversal_reason'] = f'depends on {edge_key}'
        g['edges'][key]['reversed_at'] = reversed_at

    ledger = {
        'reversal_id': 'REV_NB22_001',
        'poisoned_edge_id': edge_key,
        'original_approval_record': g['edges'][edge_key]['approval_record'],
        'affected_object_ids': sorted(affected_nodes),
        'affected_decision_ids': sorted(n for n in affected_nodes if g['nodes'][n].get('object_type') == 'decision'),
        'archived_edge_ids': [edge_key] + sorted(key for _, _, key, _ in affected_edges),
        'external_actions_to_reconcile': ['acquisition facility draw', 'FX hedge', 'integration launch'],
        'reversed_at': reversed_at,
    }
    return ledger

REVERSAL_LEDGER = reverse_dependency_cone(G)
pprint(REVERSAL_LEDGER)

## 5. Prove restoration

The proof has three parts: no contaminated object remains canonical; no active path begins at the false claim; and the active graph hash equals the clean baseline hash.

In [ ]:
restored = active_projection(G)
residual_contaminated = sorted(n for n in affected_nodes if n in restored['nodes'])
residual_dependency_count = 0 if 'CLAIM_CLEARANCE' not in restored['nodes'] else len(dependency_cone(restored, 'CLAIM_CLEARANCE')[0]) - 1
RESTORED_HASH = graph_hash(restored)

assert residual_contaminated == []
assert residual_dependency_count == 0
assert RESTORED_HASH == BASELINE_HASH
assert 'OBS_MARKET_SHARE' in restored['nodes']

REVERSAL_LEDGER['restored_graph_hash'] = RESTORED_HASH
REVERSAL_LEDGER['residual_dependency_count'] = residual_dependency_count
REVERSAL_LEDGER['independent_verifier'] = 'NB22 deterministic assertions'

print('RESTORATION PROVED')
print('Baseline hash:', BASELINE_HASH)
print('Restored hash:', RESTORED_HASH)
print('Residual dependency count:', residual_dependency_count)

## 6. Classroom debrief

Discuss:

1. Which edges in your own decision domain transmit dependency?
2. When should a dependent decision be reversed, quarantined, re-reviewed, or merely annotated?
3. Which external actions cannot be reversed by changing the graph?
4. Who owns the reversal SLA and whose P&L pays for the review hours?
5. How would access-conditioned navigation change the incident if the poisoned edge originated on the private side?
6. What would mandated erasure require if the poisoned source had to be deleted rather than archived?

**Final lesson:** reversibility is a property of the dependency model, governance record, and operating process together. A database rollback alone is not institutional reversal.